# Anexo: cálculos contrastables

Cada sección recalcula un resultado **a mano con numpy** (fórmulas escritas aquí, sin usar el bot) y lo compara con
lo que entrega el código del bot. Si algo no coincide, la celda falla con `AssertionError`.

Datos: `paquete_v2/data/` (paquete docente, verificado con su manifiesto SHA-256). Se ejecuta desde
`robot_analitico/docs/anexos/` o desde `robot_analitico/` (ajusta la ruta sola).

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import norm

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paquete_v2").is_dir())
sys.path.insert(0, str(ROOT))
DATA = ROOT / "paquete_v2" / "data"
print("Raíz del proyecto:", ROOT)

Raíz del proyecto: /home/user/RobotChechoTeoriadePortafolio/robot_analitico


## 1. Markowitz con dos activos (M3, M4)
GMV: w_A = (σ²_B − σ_AB)/(σ²_A + σ²_B − 2σ_AB). Tangente: w ∝ Σ⁻¹(μ − rf). Óptimo personal:
w_A = [(μ_A − μ_B) + γ(σ²_B − σ_AB)] / [γ(σ²_A + σ²_B − 2σ_AB)].

In [2]:
d = json.loads((DATA / "dos_activos.json").read_text())
mu, S, rf, g = np.array(d["mu"]), np.array(d["Sigma"]), d["rf"], d["gamma"]
a, b, c = S[0, 0], S[1, 1], S[0, 1]
gmv = np.array([(b - c) / (a + b - 2 * c)]); gmv = np.append(gmv, 1 - gmv[0])
z = np.linalg.solve(S, mu - rf); tan = z / z.sum()
per = np.array([((mu[0] - mu[1]) + g * (b - c)) / (g * (a + b - 2 * c))]); per = np.append(per, 1 - per[0])
mano = pd.DataFrame({"GMV": gmv, "Tangente": tan, "Personal": per}, index=d["assets"])

from src.portfolio import min_variance, max_sharpe, personal_optimum
m_, S_ = pd.Series(mu, index=d["assets"]), pd.DataFrame(S, index=d["assets"], columns=d["assets"])
bnd = [tuple(x) for x in d["bounds"]]
bot = pd.DataFrame({"GMV": min_variance(m_, S_, bnd).weights, "Tangente": max_sharpe(m_, S_, rf, bnd).weights,
                    "Personal": personal_optimum(m_, S_, g, bnd).weights})
assert np.allclose(mano.values, bot.values, atol=1e-8)
pd.concat({"a mano": mano, "bot": bot}, axis=1).round(6)

a mano                           bot                    
        GMV  Tangente  Personal       GMV  Tangente  Personal
A  0.857143  0.634146  0.571429  0.857143  0.634146  0.571429
B  0.142857  0.365854  0.428571  0.142857  0.365854  0.428571

## 2. Beta, Hamada y Ke del caso Alimentos (M7)
β = Cov(Rᵢ − rf, R_m − rf)/Var(R_m − rf); βU = βL/[1 + (1 − t)D/E]; βL' = βU[1 + (1 − t)D'/E'];
Ke_USD = rf + βL'·ERP + λ·CRP; Ke_COP = (1 + Ke_USD)(1 + π_COL)/(1 + π_USA) − 1.

In [3]:
p = json.loads((DATA / "caso_alimentos_parametros.json").read_text())
r = pd.read_csv(DATA / "caso_alimentos_retornos.csv").set_index("anio")
xi, xm = r["comparable"] - p["rf_USD"], r["referencia"] - p["rf_USD"]
beta = np.cov(xi, xm, ddof=1)[0, 1] / np.var(xm, ddof=1)
bu = beta / (1 + (1 - p["tax_supuesto"]) * p["de_comparable"])
bl = bu * (1 + (1 - p["tax_supuesto"]) * p["de_empresa"])
ke = p["rf_USD"] + bl * p["erp_madura"] + p["lambda"] * p["proxy_pais"]
ke_cop = (1 + ke) * (1 + p["inflacion_COL"]) / (1 + p["inflacion_USA"]) - 1

from src.capm import beta_ols, unlever_beta, relever_beta, cost_of_equity, nominal_rate_to_other_currency
b_bot = beta_ols(r["comparable"], r["referencia"], rf=p["rf_USD"]).beta
bl_bot = relever_beta(unlever_beta(b_bot, p["de_comparable"], p["tax_supuesto"]), p["de_empresa"], p["tax_supuesto"])
ke_bot = cost_of_equity(p["rf_USD"], bl_bot, p["erp_madura"], p["proxy_pais"], p["lambda"]).ke
assert np.allclose([beta, bl, ke, ke_cop], [b_bot, bl_bot, ke_bot, nominal_rate_to_other_currency(ke_bot, p["inflacion_COL"], p["inflacion_USA"])])
pd.Series({"beta": beta, "βU": bu, "βL empresa": bl, "Ke USD": ke, "Ke COP": ke_cop}).round(6)

beta          1.514286
βU            1.089414
βL empresa    1.620504
Ke USD        0.164128
Ke COP        0.186842
dtype: float64

## 3. Black-Litterman (M5)
π = δΣw_ref; μ_BL = [(τΣ)⁻¹ + P'Ω⁻¹P]⁻¹[(τΣ)⁻¹π + P'Ω⁻¹Q] (forma con inversas, solo para contrastar).

In [4]:
d = json.loads((DATA / "black_litterman_control.json").read_text())
S, w, P, Q, Om, tau = np.array(d["Sigma"]), np.array(d["w_ref"]), np.array(d["P"]), np.array(d["Q"]), np.array(d["Omega"]), d["tau"]
pi = d["delta"] * S @ w
inv = np.linalg.inv
mu_bl = inv(inv(tau * S) + P.T @ inv(Om) @ P) @ (inv(tau * S) @ pi + P.T @ inv(Om) @ Q)

from src.black_litterman import View, build_view_set, implied_prior, posterior
A = d["assets"]; S_ = pd.DataFrame(S, index=A, columns=A)
vs = build_view_set([View(f"V{i+1}", dict(zip(A, P[i])), Q[i], omega=Om[i, i]) for i in range(len(Q))], A, S_, tau)
bot = posterior(S_, implied_prior(S_, pd.Series(w, index=A), d["delta"]), tau, vs).posterior
assert np.allclose(mu_bl, bot.values, atol=1e-10)
pd.DataFrame({"prior π": pi, "μ_BL a mano": mu_bl, "μ_BL bot": bot.values}, index=A).round(6)

,prior π,μ_BL a mano,μ_BL bot
A,0.0228,0.0285,0.0285
B,0.0552,0.0450,0.0450


## 4. Rebalanceo de control (M6)
Valor previo, nominal negociado, costos y operaciones para volver a los pesos objetivo.

In [5]:
c = json.loads((DATA / "rebalanceo_control.json").read_text())
q, p1, wt, k = np.array(c["cantidades"], float), np.array(c["precios_nuevos"], float), np.array(c["pesos_objetivo"], float), c["costo_por_nominal"]
v_pre = q @ p1 + c["caja"]
# Sin caja mínima: V_post = V_pre − k·Σ|trade| y trade_i = w_i·V_post − q_i p_i; con dos activos y uno que vende,
# el nominal total es |trade_A| + |trade_B|; se resuelve el punto fijo.
v_post = v_pre
for _ in range(100):
    trades = wt * v_post - q * p1
    v_post = v_pre - k * np.abs(trades).sum()

from src.rebalance import Book, plan_rebalance
A = c["assets"]
plan = plan_rebalance(Book(quantities=pd.Series(q, index=A), cash=c["caja"]), pd.Series(p1, index=A), pd.Series(wt, index=A), k)
assert np.isclose(v_pre, plan.equity_before) and np.isclose(v_post, plan.equity_after) and np.allclose(trades, plan.trades_value, atol=1e-6)
pd.Series({"V_pre": v_pre, "nominal": np.abs(trades).sum(), "costos": k * np.abs(trades).sum(), "V_post": v_post}).round(4)

V_pre      10400.0
nominal     2800.0
costos         2.8
V_post     10397.2
dtype: float64

## 5. Pérdida esperada (ES) del modelo lognormal
ES = 1 − exp(M + S²/2)·Φ((q − M − S²)/S)/(1 − c), con q = M + S·z₁₋c. Se contrasta con 2 millones de simulaciones.

In [6]:
from src.forecasting import ForecastModel
from src.risk_extras import expected_shortfall_fraction
M, Sd, cnf = 0.0004 * 10, 0.015 * np.sqrt(10), 0.95
x = np.random.default_rng(0).normal(M, Sd, 2_000_000)
loss = 1 - np.exp(x)
sim = loss[loss >= np.quantile(loss, cnf)].mean()
bot = expected_shortfall_fraction(ForecastModel("lognormal", 0.0004, 0.015, 500), 10, cnf)
assert abs(bot / sim - 1) < 5e-3
pd.Series({"ES simulada": sim, "ES bot (fórmula)": bot}).round(6)

ES simulada         0.089381
ES bot (fórmula)    0.089434
dtype: float64

## 6. Desempeño frente al benchmark
Sharpe, Treynor, alfa de Jensen, error de seguimiento y razón de información con fórmulas directas.

In [7]:
rng = np.random.default_rng(5)
idx = pd.bdate_range("2022-01-03", periods=750)
bm = pd.Series(rng.normal(0.0004, 0.01, len(idx)), index=idx)
pt = 0.0002 + 1.2 * bm + pd.Series(rng.normal(0, 0.006, len(idx)), index=idx)
rf, m = 0.0001, 252
beta = np.cov(pt - rf, bm - rf, ddof=1)[0, 1] / np.var(bm, ddof=1)
mano = {"Sharpe": (pt.mean() - rf) / pt.std() * np.sqrt(m), "Treynor": (pt.mean() - rf) * m / beta,
        "alfa Jensen": (pt.mean() - rf - beta * (bm.mean() - rf)) * m,
        "razón de información": (pt - bm).mean() * m / ((pt - bm).std() * np.sqrt(m))}
from src.performance import performance
b = performance("cartera", pt, bm, rf, m)
bot = {"Sharpe": b.sharpe, "Treynor": b.treynor, "alfa Jensen": b.jensen_alpha, "razón de información": b.information_ratio}
assert np.allclose(list(mano.values()), list(bot.values()))
pd.DataFrame({"a mano": mano, "bot": bot}).round(6)

,a mano,bot
Sharpe,0.910954,0.910954
Treynor,0.159260,0.159260
alfa Jensen,0.127395,0.127395
razón de información,1.421691,1.421691


## 7. Moneda: 1 + R_COP = (1 + R_USD)(1 + R_FX)

In [8]:
idx = pd.bdate_range("2024-01-02", periods=5)
p_usd = pd.Series([100, 102, 101, 105, 104.0], index=idx)
fx = pd.Series([3900, 3950, 4010, 3980, 4000.0], index=idx)
from src.data_quality import convert_prices_base
p_cop = convert_prices_base(p_usd.to_frame("A"), "USD", "COP", fx)[0]["A"]
lhs = 1 + p_cop.pct_change().dropna()
rhs = (1 + p_usd.pct_change().dropna()) * (1 + fx.pct_change().dropna())
assert np.allclose(lhs, rhs)
pd.DataFrame({"1 + R_COP": lhs, "(1 + R_USD)(1 + R_FX)": rhs}).round(8)

,1 + R_COP,(1 + R_USD)(1 + R_FX)
2024-01-03,1.033077,1.033077
2024-01-04,1.005237,1.005237
2024-01-05,1.031826,1.031826
2024-01-08,0.995453,0.995453


Todas las celdas terminaron sin error: los cálculos a mano y los del bot coinciden.